In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    compute_batch_size,
    detect_device,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_B'

df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  MAX_BATCH=65,536  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=65,536  adaptive BATCH_SIZE=16,384  INIT_LR=0.002000  n_train=911,172  steps/epoch~55
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')

Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.53 GB / 85 GB  |  Free: 84.6 GB
Train: 911,172  Val: 260,336  Test: 130,168  Features: 12


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 38.6267  RMSE = 0.017226
Coefficients: a = -0.092294, b = -0.154601, c = -0.144622


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 16,384  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=38.6766  Gain=-0.1%  ep=100  19.5s  elapsed=0.3min
  [ 25/130] 3F+iv_lag+rho                  SSE=24.0109  Gain=+37.8%  ep=100  13.5s  elapsed=5.7min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=18.9724  Gain=+50.9%  ep=100  13.7s  elapsed=11.4min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=19.7083  Gain=+49.0%  ep=100  13.3s  elapsed=17.0min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=27.3433  Gain=+29.2%  ep=100  13.7s  elapsed=22.7min
  [125/130] 3F+vix_mom+theta+rho           SSE=35.8165  Gain=+7.3%  ep=100  13.4s  elapsed=28.3min
  [130/130] 3F+theta+vega+rho              SSE=36.8721  Gain=+4.5%  ep=100  13.9s  elapsed=29.5min

Done: 29.5 min for 130 models (avg 13.6s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,38.626713,0.000297,0.017226,0.006097,0.001861,0.002207,0.067476,None,None,None
1,3F,38.676601,0.000297,0.017237,0.007821,0.000199,0.004333,0.066271,14.6s,-0.13%,None
2,3F+vix_lag,46.606579,0.000358,0.018922,0.011231,-0.000839,0.007568,-0.125174,13.6s,-20.66%,-20.50%
3,3F+iv_lag,32.792351,0.000252,0.015872,0.010067,-0.000324,0.007050,0.208328,13.6s,15.10%,29.64%
4,3F+d_iv_lag,36.162449,0.000278,0.016668,0.009564,0.000332,0.006042,0.126968,13.2s,6.38%,-10.28%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,37.057690,0.000285,0.016873,0.008340,0.000801,0.005089,0.105355,13.3s,4.06%,-0.39%
128,3F+gamma+theta+rho,40.348080,0.000310,0.017606,0.009393,-0.000579,0.006042,0.025918,13.7s,-4.46%,-8.88%
129,3F+gamma+vega+rho,36.064465,0.000277,0.016645,0.008171,0.001020,0.004919,0.129333,13.8s,6.63%,10.62%
130,3F+theta+vega+rho,36.872089,0.000283,0.016830,0.007927,-0.000620,0.004571,0.109836,13.8s,4.54%,-2.24%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+iv_lag+gamma+rho,6,14.9880,0.010730,61.20,13.6
1,3F+vix_lag+iv_lag+d_iv_lag,6,17.8863,0.011722,53.69,13.6
2,3F+iv_lag+gamma+vega,6,17.9216,0.011734,53.60,13.5
3,3F+vix_lag+iv_lag+gamma,6,18.9724,0.012073,50.88,13.7
4,3F+vix_lag+iv_lag+vega,6,19.0365,0.012093,50.72,13.8
5,3F+iv_lag+d_iv_lag+rho,6,19.3005,0.012177,50.03,13.7
6,3F+vix_lag+iv_lag+theta,6,19.3221,0.012184,49.98,13.6
7,3F+iv_lag+gamma+theta,6,19.3421,0.012190,49.93,13.4
8,3F+vix_lag+iv_lag+rho,6,19.6645,0.012291,49.09,13.5
9,3F+iv_lag+d_iv_lag+vix_mom_lag,6,19.7083,0.012305,48.98,13.3


## Best per group (3F, +1, +2, +3)

In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): 3F
    SSE=38.6766  RMSE=0.017237  Gain=-0.13%

+1 (4F): 3F+iv_lag
    SSE=32.7924  RMSE=0.015872  Gain=15.10%

+2 (5F): 3F+iv_lag+gamma
    SSE=21.2199  RMSE=0.012768  Gain=45.06%

+3 (6F): 3F+iv_lag+gamma+rho
    SSE=14.9880  RMSE=0.010730  Gain=61.20%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 29.3 min (0.49 hr)
Models trained: 130
Best overall: 3F+iv_lag+gamma+rho (Gain=61.20%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.70 GB / 85 GB
Total training time: 29.3 min for 130 models
